# U-Net segmentation with uncertainty

This notebook trains a dropout-enabled U-Net and reproduces the four-panel display from the referenced notebook: **input**, **ground truth**, **inference**, and an **uncertainty mask**. The trained checkpoint is written to `models/unet_mc_dropout_best_model.pth`.

The uncertainty panel is the pixelwise standard deviation across Monte Carlo dropout predictions. Dropout is applied to the U-Net decoder features during training and deliberately re-enabled, while batch normalization remains in evaluation mode, during uncertainty sampling.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
import segmentation_models_pytorch as smp

# Set CME_DATA_ROOT to override the default data directory on another server.
PROJECT_ROOT = Path.cwd()
CHECKPOINT_PATH = PROJECT_ROOT / 'models' / 'unet_mc_dropout_best_model.pth'
DATA_ROOT = Path(os.environ.get('CME_DATA_ROOT', '/data/CME_reconstruction/cme_silhouette/processed_dataset_new'))
CSV_PATH = DATA_ROOT / 'file_list.csv'
TARGET_SIZE = (832, 832)  # Must match training.
THRESHOLD = 0.5
BATCH_SIZE = 12
NUM_WORKERS = 8
NUM_EPOCHS = 20
LEARNING_RATE = 1e-4
ALPHA_LOSS = 0.5
SEED = 42
DROPOUT_P = 0.20
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'Using {DEVICE}; output checkpoint: {CHECKPOINT_PATH}')

In [ ]:
def pad_to_target(img, target_height=832, target_width=832):
    # Replace NaN values with 0
    img = np.nan_to_num(img, nan=0.0)
    if img.ndim == 2:
        h, w = img.shape
        pad_h = target_height - h
        pad_w = target_width - w
        top = pad_h // 2
        bottom = pad_h - top
        left = pad_w // 2
        right = pad_w - left
        padded = np.pad(img, ((top, bottom), (left, right)), mode='constant', constant_values=0)
    elif img.ndim == 3:
        c, h, w = img.shape
        pad_h = target_height - h
        pad_w = target_width - w
        top = pad_h // 2
        bottom = pad_h - top
        left = pad_w // 2
        right = pad_w - left
        padded = np.pad(img, ((0, 0), (top, bottom), (left, right)), mode='constant', constant_values=0)
    else:
        raise ValueError('Unsupported image dimensions')
    return padded

class ProcessedSegmentationDataset(Dataset):
    def __init__(self, csv_path, root_dir):
        self.files = pd.read_csv(csv_path)
        self.root_dir = Path(root_dir)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        image, mask, _ = self.load(idx)
        return torch.from_numpy(image)[None], torch.from_numpy(mask)[None]

    def load(self, idx):
        if not 0 <= idx < len(self.files):
            raise IndexError(f'idx must be in [0, {len(self.files) - 1}].')
        sample_path = self.root_dir / self.files.iloc[idx]['file_path']
        sample = np.load(sample_path, allow_pickle=True).item()
        image = pad_to_target(sample['input'], target_height=TARGET_SIZE[0], target_width=TARGET_SIZE[1])
        mask = np.asarray(sample['mask']).squeeze()
        mask = pad_to_target((mask > 0).astype(np.float32), target_height=TARGET_SIZE[0], target_width=TARGET_SIZE[1])
        return image, mask, sample_path

assert CSV_PATH.exists(), f'Dataset index not found: {CSV_PATH}'
dataset = ProcessedSegmentationDataset(CSV_PATH, DATA_ROOT)
indices = list(range(len(dataset)))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=SEED)
train_loader = DataLoader(Subset(dataset, train_idx), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(Subset(dataset, val_idx), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

def load_sample(idx):
    return dataset.load(idx)


In [ ]:
class MCDropoutUnet(smp.Unet):
    """Trainable U-Net with spatial MC dropout on decoder features."""
    def __init__(self, dropout_p=DROPOUT_P, **kwargs):
        super().__init__(**kwargs)
        self.mc_dropout = nn.Dropout2d(p=dropout_p)

    def forward(self, x):
        self.check_input_shape(x)
        features = self.encoder(x)
        decoder_output = self.decoder(features)
        decoder_output = self.mc_dropout(decoder_output)
        return self.segmentation_head(decoder_output)

# U-Net with a ResNet-18 encoder and one input/output channel.
model = MCDropoutUnet(
    encoder_name='resnet18', encoder_weights='imagenet',
    in_channels=1, classes=1, activation=None,
).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

def dice_loss(logits, targets, smooth=1e-6):
    probabilities = torch.sigmoid(logits)
    intersection = (probabilities * targets).sum(dim=(1, 2, 3))
    union = probabilities.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    return (1 - (2 * intersection + smooth) / (union + smooth)).mean()

def combined_loss(logits, targets):
    return ALPHA_LOSS * nn.BCEWithLogitsLoss()(logits, targets) + (1 - ALPHA_LOSS) * dice_loss(logits, targets)

print(f'MC dropout p={DROPOUT_P}; model will be saved to {CHECKPOINT_PATH}')

In [ ]:
# Run this cell once to train. Dropout is active because model.train() is used.
# The best validation-loss checkpoint is saved for future uncertainty-only runs.
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
best_val_loss = float('inf')
for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss_total = 0.0
    for inputs, masks in train_loader:
        inputs, masks = inputs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss = combined_loss(logits, masks)
        loss.backward()
        optimizer.step()
        train_loss_total += loss.item() * inputs.size(0)

    model.eval()
    val_loss_total = 0.0
    with torch.inference_mode():
        for inputs, masks in val_loader:
            inputs, masks = inputs.to(DEVICE), masks.to(DEVICE)
            val_loss_total += combined_loss(model(inputs), masks).item() * inputs.size(0)
    train_loss = train_loss_total / len(train_loader.dataset)
    val_loss = val_loss_total / len(val_loader.dataset)
    print(f'Epoch {epoch + 1:02d}/{NUM_EPOCHS}: train loss={train_loss:.5f}, val loss={val_loss:.5f}')
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(), 'loss': val_loss,
                    'dropout_p': DROPOUT_P}, CHECKPOINT_PATH)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'Loaded best MC-dropout checkpoint (val loss={checkpoint["loss"]:.5f}).')

In [ ]:
def enable_mc_dropout(module):
    """Enable dropout while preserving eval-mode BatchNorm statistics."""
    module.eval()
    dropout_layers = [child for child in module.modules()
                      if isinstance(child, (nn.Dropout, nn.Dropout2d, nn.Dropout3d, nn.AlphaDropout))]
    if not dropout_layers:
        raise RuntimeError(
            'No dropout layers were found. MC dropout requires a U-Net trained with dropout. '
            'Use the matching dropout-enabled architecture/checkpoint; do not add dropout only at inference.'
        )
    for layer in dropout_layers:
        layer.train()
    return len(dropout_layers)

@torch.inference_mode()
def mc_predict(input_tensor, n_samples=100):
    if n_samples < 2:
        raise ValueError('n_samples must be at least 2 to calculate a standard deviation.')
    enable_mc_dropout(model)
    probabilities = torch.stack([torch.sigmoid(model(input_tensor)) for _ in range(n_samples)])
    model.eval()
    return probabilities.mean(dim=0), probabilities.std(dim=0, unbiased=False)

In [ ]:
def displayTestcaseWithUncertainty(idx, n_samples=100, string=''):
    """Show input, ground truth, mean inference, and MC-dropout uncertainty."""
    image, ground_truth, sample_path = load_sample(idx)
    input_tensor = torch.from_numpy(image)[None, None].to(DEVICE)

    model.eval()
    with torch.inference_mode():
        deterministic_probability = torch.sigmoid(model(input_tensor))[0, 0].cpu().numpy()
    mean_probability, uncertainty = mc_predict(input_tensor, n_samples=n_samples)
    uncertainty = uncertainty[0, 0].cpu().numpy()

    fig, axes = plt.subplots(1, 4, figsize=(18, 5), constrained_layout=True)
    axes[0].imshow(image, cmap='gray')
    axes[0].set_title('input')
    axes[1].imshow(ground_truth, cmap='gray', vmin=0, vmax=1)
    axes[1].set_title('ground truth')
    axes[2].imshow(deterministic_probability, cmap='gray', vmin=0, vmax=1)
    axes[2].contour(deterministic_probability >= THRESHOLD, levels=[0.5], colors='lime', linewidths=0.7)
    axes[2].set_title('inference (probability)')
    uncertainty_plot = axes[3].imshow(uncertainty, cmap='plasma', vmin=0)
    axes[3].set_title(f'uncertainty (MC std, n={n_samples})')
    fig.colorbar(uncertainty_plot, ax=axes[3], fraction=0.046, pad=0.04, label='probability std. dev.')
    for axis in axes:
        axis.axis('off')
    fig.suptitle(string or f'idx={idx} — {sample_path.name}')
    plt.show()
    return {'input': image, 'ground_truth': ground_truth,
            'inference': deterministic_probability, 'mean_probability': mean_probability[0, 0].cpu().numpy(),
            'uncertainty_mask': uncertainty}


In [ ]:
# Pick any valid index from file_list.csv. The returned dictionary includes the raw uncertainty mask.
result = displayTestcaseWithUncertainty(idx=0, n_samples=100, string='CME U-Net uncertainty')
uncertainty_mask = result['uncertainty_mask']
# np.save('uncertainty_mask_idx_0.npy', uncertainty_mask)  # optional export
